# 🦷 Calculus Curation UI (Gold Standard Subset)

This tool allows you to manually inspect raw dental meshes and flag them as **Clean** or **Has Calculus**. 
The results are saved to a CSV ledger (`curation_ledger.csv`), which is then used to filter the dataset generation pipeline to ensure a mathematically perfect baseline.

In [ ]:
!pip install trimesh scipy plotly ipywidgets pandas

In [ ]:
import os
import json
import csv
import trimesh
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
import ipywidgets as widgets
from IPython.display import display, clear_output
from pathlib import Path

# Colab-specific initialization for widgets
try:
    from google.colab import output
    output.enable_custom_widget_manager()
except ImportError:
    pass

RAW_DATASET_ROOT = "data_part_1"
LEDGER_PATH = "curation_ledger.csv"

def load_ledger():
    if os.path.exists(LEDGER_PATH):
        return pd.read_csv(LEDGER_PATH).set_index('patient_id').to_dict('index')
    return {}

def save_to_ledger(patient_id, status):
    file_exists = os.path.exists(LEDGER_PATH)
    # Check if patient already exists to update or append
    ledger = load_ledger()
    ledger[patient_id] = {'status': status}
    
    df = pd.DataFrame.from_dict(ledger, orient='index')
    df.index.name = 'patient_id'
    df.to_csv(LEDGER_PATH)

def get_raw_scans(root_dir):
    root_path = Path(root_dir)
    scans = []
    if not root_path.exists():
        return scans
        
    for obj_file in root_path.rglob("*.obj"):
        expected_json = obj_file.parent / f"{obj_file.stem}.json"
        if expected_json.exists():
            json_file = expected_json
        else:
            jsons = [f for f in obj_file.parent.glob("*.json") if "__kpt" not in f.name]
            if not jsons:
                continue
            json_file = jsons[0]
            
        patient_id = obj_file.parent.name
        scans.append({'id': patient_id, 'obj': str(obj_file), 'json': str(json_file)})
        
    return sorted(scans, key=lambda x: x['id'])

# --- State & Data ---
all_scans = get_raw_scans(RAW_DATASET_ROOT)
ledger = load_ledger()
current_index = 0

if not all_scans:
    print(f"No data found in {RAW_DATASET_ROOT}. Please extract your data first.")
else:
    # --- GUI Components ---
    lbl_info = widgets.HTML(value="<b>Loading...</b>")
    
    btn_clean = widgets.Button(description='✅ Clean', button_style='success')
    btn_calculus = widgets.Button(description='❌ Has Calculus', button_style='danger')
    btn_prev = widgets.Button(description='⬅️ Previous', button_style='info')
    btn_next = widgets.Button(description='➡️ Next', button_style='info')
    
    status_output = widgets.Output()
    
    fig_widget = go.FigureWidget()
    fig_widget.layout = dict(
        scene=dict(xaxis=dict(visible=False), yaxis=dict(visible=False), zaxis=dict(visible=False), aspectmode='data'),
        margin=dict(l=0, r=0, b=0, t=0),
        height=600
    )
    
    def update_ui():
        global current_index, ledger
        if current_index < 0:
            current_index = 0
        if current_index >= len(all_scans):
            current_index = len(all_scans) - 1
            
        scan = all_scans[current_index]
        pat_id = scan['id']
        
        # Check current status in ledger
        ledger = load_ledger()
        status = ledger.get(pat_id, {}).get('status', 'Unreviewed')
        
        color = "black"
        if status == "Clean": color = "green"
        elif status == "Has Calculus": color = "red"
        
        lbl_info.value = f"<h3>Scan {current_index + 1} / {len(all_scans)} | Patient ID: {pat_id}</h3><p>Current Status: <b style='color:{color}'>{status}</b></p>"
        
        with status_output:
            clear_output(wait=True)
            print(f"Rendering {pat_id}...")
            
        try:
            mesh = trimesh.load(scan['obj'], process=False)
            with open(scan['json'], 'r') as f:
                label_data = json.load(f)
            labels = np.array(label_data.get('labels', np.zeros(len(mesh.vertices))))
            
            colors = np.ones((len(mesh.vertices), 3)) * 255
            colors[labels == 0] = [230, 150, 160]   # Gum
            colors[labels > 0] = [240, 240, 240]    # Teeth
            vertex_colors = [f'rgb({int(c[0])}, {int(c[1])}, {int(c[2])})' for c in colors]
            
            trace = go.Mesh3d(
                x=mesh.vertices[:, 0], y=mesh.vertices[:, 1], z=mesh.vertices[:, 2],
                i=mesh.faces[:, 0], j=mesh.faces[:, 1], k=mesh.faces[:, 2],
                vertexcolor=vertex_colors, showscale=False,
                lighting=dict(ambient=0.5, diffuse=0.8, specular=0.1, roughness=0.6)
            )
            
            with fig_widget.batch_update():
                fig_widget.data = []
                fig_widget.add_trace(trace)
                
            with status_output:
                clear_output(wait=True)
        except Exception as e:
            with status_output:
                clear_output(wait=True)
                print(f"Error loading mesh: {e}")

    def mark_clean(b):
        global current_index
        save_to_ledger(all_scans[current_index]['id'], "Clean")
        current_index += 1
        update_ui()

    def mark_calculus(b):
        global current_index
        save_to_ledger(all_scans[current_index]['id'], "Has Calculus")
        current_index += 1
        update_ui()
        
    def go_prev(b):
        global current_index
        current_index -= 1
        update_ui()
        
    def go_next(b):
        global current_index
        current_index += 1
        update_ui()

    btn_clean.on_click(mark_clean)
    btn_calculus.on_click(mark_calculus)
    btn_prev.on_click(go_prev)
    btn_next.on_click(go_next)
    
    controls = widgets.HBox([btn_prev, btn_clean, btn_calculus, btn_next])
    ui = widgets.VBox([lbl_info, controls, status_output, fig_widget])
    
    # Initialize first scan
    display(ui)
    update_ui()
